# 04 – Pilot λ_roi dan GAN (CP4, D2, FR-06)

GAN kondisional-kelas bergaya StyleGAN2 dengan D_roi (decisions.yaml D8). λ_roi dipilih dari **FID pada pasien validasi** di dalam lipatan latih (rep 0, lipatan 0); kriteria ditetapkan di konfigurasi sebelum eksperimen. Citra uji tidak dipakai.

**Versi 2:** pelatihan memakai gradient accumulation (`gan.micro_batch`, batch efektif tetap `gan.batch_size`) dan D_roi setengah lebar D (`gan.droi_d_channels`). Tetapkan `gan.micro_batch` dan `gan.total_kimg` dari `benchmark_gan.json` (notebook 00) **sebelum** menjalankan pilot, dan catat di `LOG.md`.

**Notebook mandiri (Versi 2):** tanpa folder `src`, tanpa Docker. Semua kode ada di sel-sel notebook ini. Kode ditulis dari Versi 1 dan **belum dijalankan** di mesin penulis (tanpa torch/GPU); jalankan dari atas ke bawah di mesin RTX 3070 Ti. Hasil yang tidak dapat dihitung dilaporkan apa adanya, bukan diisi.

## 1. Pustaka tambahan (opsional)

Inception-v3 (FID) dan AlexNet LPIPS mengunduh bobot pada pemakaian pertama (butuh internet).

In [ ]:
INSTALL = False      # True hanya sekali, di venv yang sama dengan notebook 00 (torch CUDA sudah terpasang di sana)

if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-image", "scikit-learn", "scipy", "pandas", "matplotlib", "lpips"])

## 2. Impor, konfigurasi, utilitas, kebocoran, dan pemeriksaan checkpoint

In [ ]:
from __future__ import annotations

import contextlib, copy, datetime as dt, gc, hashlib, importlib, io, json, math, os, platform, random, re, shutil, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import yaml
from IPython.display import display, Image

# expandable_segments mengurangi fragmentasi memori, tetapi belum didukung di Windows.
if platform.system() != "Windows":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn
import torch.nn.functional as F


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "configs" / "base.yaml").exists():
            return q
    raise FileNotFoundError("configs/base.yaml tidak ditemukan dari direktori kerja ke atas")


ROOT = find_root()
os.chdir(ROOT)

_ENV = {"LARYNX_DATASET": "dataset", "LARYNX_OUTPUTS": "outputs", "LARYNX_HF_CACHE": "hf_cache"}


def deep_merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out


def _load(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = cfg.pop("_base", None)
    return deep_merge(_load(path.parent / base), cfg) if base else cfg


def load_merged(path) -> dict:
    cfg = _load(Path(path))
    for env, key in _ENV.items():
        if os.environ.get(env):
            cfg["paths"][key] = os.environ[env]
    for key, val in cfg["paths"].items():           # jalur relatif -> mutlak terhadap folder Versi 2
        if not Path(val).is_absolute():
            cfg["paths"][key] = str((ROOT / val).resolve())
    return cfg


def P(cfg: dict, *parts: str) -> Path:
    return Path(cfg["paths"]["outputs"]).joinpath(*parts)


def config_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()


def job_seed(base_seed: int, *parts) -> int:
    key = "|".join([str(base_seed), *map(str, parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def atomic_write_bytes(path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def atomic_write_json(path, obj) -> None:
    atomic_write_bytes(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8"))


def atomic_torch_save(obj, path) -> None:
    buf = io.BytesIO(); torch.save(obj, buf); atomic_write_bytes(path, buf.getvalue())


def is_done(job_dir, cfg_hash: str) -> bool:
    done = Path(job_dir) / "DONE"
    return done.exists() and done.read_text(encoding="utf-8").strip() == cfg_hash


def mark_done(job_dir, cfg_hash: str) -> None:
    atomic_write_bytes(Path(job_dir) / "DONE", cfg_hash.encode("utf-8"))


def git_commit() -> str:
    # Versi 2 tanpa Docker: hash commit dari git (bila ada), ditambah -dirty bila ada perubahan belum di-commit.
    try:
        c = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=15).stdout.strip()
        d = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"], capture_output=True, text=True, timeout=15).stdout.strip()
        return (c or "tanpa-commit") + ("-dirty" if d and c else "")
    except Exception:  # noqa: BLE001
        return "tanpa-commit"


GIT_COMMIT = git_commit()
CFG_PATH = "configs/base.yaml"
cfg = load_merged(CFG_PATH)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = P(cfg, "cp0")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT   :", ROOT)
print("dataset:", cfg["paths"]["dataset"])
print("outputs:", cfg["paths"]["outputs"])
print("device :", DEV, "| commit:", GIT_COMMIT, "| config_hash:", config_hash(cfg)[:12])

def free_gpu() -> None:
    # Pengganti 'proses baru per job' di Versi 1: dipanggil di akhir tiap job agar memori GPU kembali bersih.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


WORK_DIR = Path(cfg["paths"]["outputs"]) / "_workers"


def install_module(name: str, src: str):
    # Windows memakai 'spawn': fungsi/kelas yang dipakai proses anak harus berada di modul yang dapat diimpor. Modul dibuat
    # saat notebook berjalan (satu-satunya sumber kodenya ada di sel notebook ini) lalu diimpor, termasuk oleh proses anak.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p = WORK_DIR / f"{name}.py"
    changed = (not p.exists()) or p.read_text(encoding="utf-8") != src
    if changed:
        p.write_text(src, encoding="utf-8")
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod) if changed and name in sys.modules else mod

In [ ]:
# ---------------------------------------------------------------- checkpoint, persetujuan manusia, LOG.md
# Pengganti checkpoint.py dan approve.py Versi 1. Persetujuan MANUSIA tidak pernah dipanggil otomatis oleh notebook.
LULUS, GAGAL, MENUNGGU, TIDAK_TERSEDIA = "LULUS", "GAGAL", "MENUNGGU", "TIDAK_TERSEDIA"
DEC_PATH, BASE_PATH, LOG_PATH = ROOT / "configs" / "decisions.yaml", ROOT / "configs" / "base.yaml", ROOT / "LOG.md"


def file_sha256(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_manifest(out_dir, files, extra=None) -> Path:
    out_dir = Path(out_dir)
    entries = {}
    for f in files:
        p = Path(f)
        if not p.exists():
            raise FileNotFoundError(f"Keluaran checkpoint tidak ditemukan: {p}")
        key = str(p.relative_to(out_dir)) if p.is_relative_to(out_dir) else str(p)
        entries[key] = {"sha256": file_sha256(p), "bytes": p.stat().st_size}
    mp = out_dir / "manifest.json"
    atomic_write_json(mp, {"files": entries, **(extra or {})})
    return mp


def _item(nama: str, status: str, detail: str) -> dict:
    return {"item": nama, "status": status, "detail": detail}


def _review_item(out_dir) -> dict:
    out_dir = Path(out_dir)
    ok = (out_dir / "REVIEWED").exists()
    return _item("Tinjauan orang kedua (berkas REVIEWED)", LULUS if ok else MENUNGGU,
                 "ada" if ok else f"setelah meninjau hasil: approve_review('{out_dir.name.upper()}', 'NAMA') (RULES §3)")


def _flag_item(out_dir, name: str, nama: str, how: str) -> dict:
    ok = (Path(out_dir) / name).exists()
    return _item(nama, LULUS if ok else MENUNGGU, "ada" if ok else how)


def _decision_item(d: str) -> dict:
    with open(DEC_PATH, "r", encoding="utf-8") as f:
        dec = yaml.safe_load(f)[d]
    st = dec["status"]
    status = {"diputuskan": LULUS, "ditunda": LULUS, "menunggu_konfirmasi": MENUNGGU}.get(st, GAGAL)
    return _item(f"{d} {st}", status, str(dec.get("isi", "")))


def _finish(cp: str, out_dir, items: list, files: list, extra=None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    blocking = [i for i in items if i["status"] in (GAGAL, MENUNGGU)]
    rep = {"checkpoint": cp, "lulus": not blocking, "items": items, "penghambat": [i["item"] for i in blocking]}
    atomic_write_json(out_dir / f"{cp.lower()}_check.json", rep)
    for i in items:
        print(f"[{i['status']:>15}] {i['item']} -- {i['detail']}")
    if rep["lulus"]:
        write_manifest(out_dir, [out_dir / f"{cp.lower()}_check.json", *[f for f in files if Path(f).exists()]],
                       extra={"checkpoint": cp, "git_commit": GIT_COMMIT, **(extra or {})})
        print(f"\n{cp} LULUS. manifest.json ditulis. Tag git '{cp.lower()}' dapat dibuat (setelah tinjauan orang kedua).")
    else:
        print(f"\n{cp} BELUM LULUS. Penghambat: {rep['penghambat']}\nmanifest.json TIDAK ditulis.")
    return rep


def log_row(who: str, what: str, hasil: str = "disetujui", keputusan: str = "via notebook") -> None:
    line = f"| {dt.date.today().isoformat()} | {who} | {what} | {hasil} | {keputusan} |\n"
    if LOG_PATH.exists():
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line)


def approve_flag(cp: str, name: str, by: str, what: str) -> None:
    d = P(load_merged(BASE_PATH), cp.lower())
    d.mkdir(parents=True, exist_ok=True)
    (d / name).write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, f"{what} ({cp})")
    print("ditulis", d / name)


def approve_review(cp: str, by: str) -> None:
    approve_flag(cp, "REVIEWED", by, "Tinjauan orang kedua")


def approve_decisions(ids: list, by: str) -> None:
    text = DEC_PATH.read_text(encoding="utf-8")
    for d in ids:
        pat = re.compile(rf"(^{d}:\n  status: )menunggu_konfirmasi", re.M)
        if not pat.search(text):
            print(f"{d}: tidak berstatus menunggu_konfirmasi (dilewati)")
            continue
        text = pat.sub(rf"\1diputuskan\n  dikonfirmasi_oleh: {by}\n  tanggal_konfirmasi: {dt.date.today().isoformat()}", text)
        log_row(by, f"Konfirmasi keputusan {d}")
        print(f"{d}: diputuskan oleh {by}")
    DEC_PATH.write_text(text, encoding="utf-8")


def approve_roi(by: str) -> None:
    t = BASE_PATH.read_text(encoding="utf-8")
    t2 = re.sub(r"(  roi:.*?\n    locked: )false", r"\1true", t, count=1, flags=re.S)
    if t2 == t and "locked: true" not in t:
        raise SystemExit("preprocess.roi.locked tidak ditemukan di base.yaml")
    BASE_PATH.write_text(t2, encoding="utf-8")
    d = P(load_merged(BASE_PATH), "cp2")
    d.mkdir(parents=True, exist_ok=True)
    (d / "ROI_APPROVED").write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, "Parameter Frangi dikunci setelah pemeriksaan visual 50 citra (FR-03)")
    print("preprocess.roi.locked=true; ditulis", d / "ROI_APPROVED")

In [ ]:
# ---------------------------------------------------------------- pengendalian kebocoran L1-L6 (dari leakage.py Versi 1)
class LeakageError(AssertionError):
    pass


def check_l1(splits: dict) -> bool:
    # L1: tidak ada pasien di latih/validasi dan uji sekaligus; tiap pasien diuji tepat sekali per ulangan.
    allp = set(splits["all_patients"])
    for r in splits["repeats"]:
        tested = []
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if tr & te:
                raise LeakageError(f"L1: pasien latih juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(tr & te)[:5]}")
            if va & te:
                raise LeakageError(f"L1: pasien validasi juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(va & te)[:5]}")
            if tr & va:
                raise LeakageError(f"L1: pasien latih juga di validasi (rep {r['rep']} lipatan {f['fold']})")
            if tr | va | te != allp:
                raise LeakageError(f"L1: gabungan partisi != seluruh pasien (rep {r['rep']} lipatan {f['fold']})")
            tested += list(te)
        if sorted(tested) != sorted(allp):
            raise LeakageError(f"L1: pada rep {r['rep']} tiap pasien harus diuji tepat sekali")
    return True


def check_l5(splits: dict, resolution_of: dict) -> bool:
    # L5 (S7): semua pasien latih/validasi berresolusi mayoritas, semua pasien uji bukan.
    major = splits["major_resolution"]
    for r in splits["repeats"]:
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if (tr | va) & te or tr & va:
                raise LeakageError(f"L5: partisi S7 saling tumpang tindih (rep {r['rep']})")
            bad = [p for p in tr | va if resolution_of[p] != major]
            if bad:
                raise LeakageError(f"L5: pasien non-mayoritas di latih/validasi: {sorted(bad)[:5]}")
            bad = [p for p in te if resolution_of[p] == major]
            if bad:
                raise LeakageError(f"L5: pasien resolusi mayoritas di uji S7: {sorted(bad)[:5]}")
    return True


def check_splits(splits: dict, resolution_of=None) -> bool:
    if splits.get("kind") == "s7":
        if resolution_of is None:
            raise LeakageError("L5: peta resolusi pasien diperlukan untuk memeriksa split S7")
        return check_l5(splits, resolution_of)
    return check_l1(splits)


def check_l6(cfg: dict) -> bool:
    # L6: metrik struktur pembuluh memakai filter berbeda dari pembuat ROI.
    if cfg["gan"]["quality"]["vessel_filter"].lower() == cfg["preprocess"]["roi"]["filter"].lower():
        raise LeakageError("L6: filter metrik pembuluh sama dengan filter pembuat ROI")
    return True


def check_l3_job(job_patients: dict, split_fold: dict, synthetic_source_patients=None) -> bool:
    # L3: pasien yang dipakai job sama dengan partisi split; citra sintetis hanya bersumber dari pasien latih.
    test = set(split_fold["test"])
    used = set(job_patients["train"]) | set(job_patients["val"])
    if used & test:
        raise LeakageError(f"L3: job memakai pasien uji: {sorted(used & test)[:5]}")
    if set(job_patients["train"]) != set(split_fold["train"]) or set(job_patients["val"]) != set(split_fold["val"]):
        raise LeakageError("L3: daftar pasien job berbeda dari berkas split")
    if set(job_patients["test"]) != test:
        raise LeakageError("L3: daftar pasien uji job berbeda dari berkas split")
    if synthetic_source_patients is not None and not set(synthetic_source_patients) <= set(split_fold["train"]):
        raise LeakageError("L3: citra sintetis bersumber dari pasien non-latih")
    return True


def check_l3_dir(jobs_root, scenario: str, splits: dict) -> int:
    n = 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            pj = Path(jobs_root) / scenario / f"rep{r['rep']}_fold{f['fold']}" / "patients.json"
            if pj.exists():
                check_l3_job(json.loads(pj.read_text(encoding="utf-8")), f)
                n += 1
    return n


class GuardedArray:
    # L2: pembungkus larik yang menolak akses ke baris terlarang (indeks uji) dan penyerahan seluruh larik.
    def __init__(self, arr, forbidden):
        self._arr = arr
        self._forbidden = np.asarray(sorted(set(map(int, forbidden))), dtype=np.int64)
        self.accessed = set()

    def __len__(self):
        return len(self._arr)

    @property
    def shape(self):
        return self._arr.shape

    def __array__(self, *a, **k):
        raise LeakageError("L2: seluruh larik diminta pada fase fit; gunakan indeks latih/validasi saja")

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            pos = np.arange(len(self._arr))[idx]
        else:
            idx_a = np.asarray(idx)
            pos = np.flatnonzero(idx_a) if idx_a.dtype == bool else idx_a.ravel().astype(np.int64)
        if np.intersect1d(pos, self._forbidden).size:
            raise LeakageError("L2: fase fit mengakses baris data uji")
        self.accessed.update(int(p) for p in pos)
        return np.asarray(self._arr[idx])

In [ ]:
# ---------------------------------------------------------------- keluaran job, pembacaan split, penjalan job (dari jobio.py, splits.py, jobs.py)
def job_dir(cfg: dict, scenario: str, rep: int, fold: int) -> Path:
    return P(cfg, "jobs", scenario, f"rep{rep}_fold{fold}")


def resolution_map(cfg: dict) -> dict:
    a = pd.read_csv(P(cfg, "cp1", "audit.csv"))
    return dict(zip(a.patient_uid, a.resolution))


def write_run_info(d: Path, cfg: dict, cfg_hash: str, seed: int, extra: dict) -> None:
    atomic_write_json(d / "run_info.json", {
        "scenario": cfg.get("scenario"), "config_hash": cfg_hash, "seed": seed, "git_commit": GIT_COMMIT,
        "torch": torch.__version__, "platform": platform.platform(), **extra})


def write_preds(d: Path, name: str, meta: pd.DataFrame, idx: np.ndarray, probs: np.ndarray, thr: float) -> pd.DataFrame:
    # preds_<name>_images.csv (per citra) dan preds_<name>_patients.csv (rata-rata per pasien, FR-09)
    df = pd.DataFrame({"idx": idx, "patient_uid": meta.patient_uid.to_numpy()[idx],
                       "label": meta.label.to_numpy()[idx], "prob_malignant": probs})
    atomic_write_bytes(d / f"preds_{name}_images.csv", df.to_csv(index=False).encode("utf-8"))
    pat = df.groupby("patient_uid").agg(label=("label", "first"), prob_malignant=("prob_malignant", "mean"),
                                        n_images=("idx", "size")).reset_index()
    pat["pred"] = (pat.prob_malignant >= thr).astype(int)
    atomic_write_bytes(d / f"preds_{name}_patients.csv", pat.to_csv(index=False).encode("utf-8"))
    return pat


def balanced_logloss(y: np.ndarray, p1: np.ndarray, eps: float = 1e-7) -> float:
    p1 = np.clip(p1, eps, 1 - eps)
    losses = [(-np.log(1 - p1[y == 0])).mean() if (y == 0).any() else None,
              (-np.log(p1[y == 1])).mean() if (y == 1).any() else None]
    return float(np.mean([v for v in losses if v is not None]))


# ---------------------------------------------------------------- split: baca
def _split_name(splits_name: str):
    return ("splits.json", "fold_summary.csv") if splits_name == "main" else (f"splits_{splits_name}.json", f"fold_summary_{splits_name}.csv")


def load_splits(cfg: dict, splits_name=None) -> dict:
    fj, _ = _split_name(splits_name or cfg.get("splits_name", "main"))
    return json.loads(P(cfg, "splits", fj).read_text(encoding="utf-8"))


def get_fold(splits: dict, rep: int, fold: int) -> dict:
    return splits["repeats"][rep]["folds"][fold]


# ---------------------------------------------------------------- penjalan job (pengganti jobs.py: pemanggilan fungsi, bukan subproses)
def job_status(cfg: dict, scenario: str) -> list:
    rows = []
    for r in load_splits(cfg)["repeats"]:
        for f in r["folds"]:
            d = job_dir(cfg, scenario, r["rep"], f["fold"])
            rows.append({"rep": r["rep"], "fold": f["fold"], "DONE": (d / "DONE").exists(), "ada_last_pt": (d / "last.pt").exists()})
    return rows


def _selected(splits: dict, reps, folds):
    for r in splits["repeats"]:
        if reps is not None and r["rep"] not in reps:
            continue
        for f in r["folds"]:
            if folds is None or f["fold"] in folds:
                yield r["rep"], f["fold"]


def run_job_safe(label: str, fn, cfg: dict, attempts: int = 2):
    """Jalankan satu job. Hanya RuntimeError (termasuk CUDA OOM) diulang sekali setelah memori GPU dibersihkan; job yang gagal
    dua kali menghentikan antrean dan HARUS dicatat di LOG.md (RULES §4). Tidak ada seed diganti atau lipatan dibuang."""
    log = P(cfg, "jobs", "launch.log")
    log.parent.mkdir(parents=True, exist_ok=True)
    err = None
    for a in range(1, attempts + 1):
        t0 = time.time()
        try:
            res = fn()
            ok = True
        except RuntimeError as e:
            res, ok, err = None, False, repr(e)[:300]
        free_gpu()
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(f"{dt.datetime.now().isoformat(timespec='seconds')}\t{label}\tpercobaan={a}\t{'OK' if ok else 'GAGAL ' + str(err)}\t{time.time() - t0:.0f}s\n")
        if ok:
            return res
    raise RuntimeError(f"Job gagal {attempts}x berturut-turut: {label}. Antrean dihentikan. Catat penyebabnya di LOG.md sebelum dilanjutkan (RULES §4). Terakhir: {err}")


def launch_scenario(config: str, fn, reps=None, folds=None) -> None:
    # satu job per (rep, lipatan): fn(cfg, rep, fold)
    cfg = load_merged(config)
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{cfg['scenario']}/rep{r}_fold{k}", lambda r=r, k=k: fn(load_merged(config), r, k), cfg)


def launch_gan_chain(config: str, gan_fn, clf_fn, ratios=None, reps=None, folds=None) -> None:
    # S5/S6/S7: GAN lipatan k, lalu klasifikator tiap rasio (urutan wajib, PRD §8). ratios=[None] untuk S7.
    cfg = load_merged(config)
    sc = cfg["scenario"]
    ratios = ratios if ratios is not None else cfg["gan"]["ratios"]
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{sc}_gan/rep{r}_fold{k}", lambda r=r, k=k: gan_fn(load_merged(config), r, k), cfg)
        for ra in ratios:
            def _clf(r=r, k=k, ra=ra):
                c = load_merged(config)
                if ra is not None:
                    c["synthetic"]["ratio"] = ra
                return clf_fn(c, r, k)
            tag = f"_r{int(round(ra * 100))}" if ra is not None else ""
            run_job_safe(f"{sc}{tag}/rep{r}_fold{k}", _clf, cfg)

In [ ]:
# ---------------------------------------------------------------- pembacaan cache memmap + Dataset (dari data.py Versi 1)
DATA_SRC = r'''
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2


class Cache:
    # images/roi dibuka MALAS dengan memmap dan TIDAK ikut ter-pickle: pickle np.memmap menyalin seluruh isinya
    # (2,2 GB x jumlah pekerja DataLoader di Windows/spawn). Tiap proses membuka ulang memmap sendiri.
    def __init__(self, cache_dir):
        self.dir = Path(cache_dir)
        self.hash = (self.dir / "DONE").read_text(encoding="utf-8").strip()
        self.meta = pd.read_csv(self.dir / "meta.csv")
        self._images = None
        self._roi = None

    @property
    def images(self):
        if self._images is None:
            self._images = np.load(self.dir / "images.npy", mmap_mode="r")
        return self._images

    @property
    def roi(self):
        if self._roi is None:
            self._roi = np.load(self.dir / "roi.npy", mmap_mode="r")
        return self._roi

    def __getstate__(self):
        st = self.__dict__.copy()
        st["_images"] = None
        st["_roi"] = None
        return st

    def indices_of(self, patients):
        return self.meta.index[self.meta.patient_uid.isin(set(patients))].to_numpy()

    @property
    def labels(self):
        return self.meta.label.to_numpy()


def build_transform(cfg, train):
    size, aug, pp = cfg["model"]["input_size"], cfg["augment"], cfg["preprocess"]
    t = []
    if train:
        t.append(v2.RandomResizedCrop(size, scale=tuple(aug["crop_scale"]), ratio=tuple(aug["crop_ratio"]), antialias=True))
        if aug["hflip"]:
            t.append(v2.RandomHorizontalFlip())
        if aug["vflip"]:
            t.append(v2.RandomVerticalFlip())
        if aug["rotation_deg"]:
            t.append(v2.RandomRotation(aug["rotation_deg"]))
    else:
        t.append(v2.Resize((size, size), antialias=True))
    t += [v2.ToDtype(torch.float32, scale=True), v2.Normalize(pp["imagenet_mean"], pp["imagenet_std"])]
    return v2.Compose(t)


class ImageDS(Dataset):
    # (tensor ternormalisasi ImageNet, label, indeks cache), dilabeli ganas dan indeks -1 untuk sintetis.
    # synthetic: None | array uint8 (N,H,W,3) | (jalur .npy, indeks terpilih): bentuk terakhir dibuka dengan memmap di tiap
    # proses (tidak ikut ter-pickle ke pekerja DataLoader).
    def __init__(self, cache, idx, cfg, train, synthetic=None):
        self.cache, self.idx = cache, np.asarray(idx)
        self.syn, self.syn_path, self.syn_idx, self._syn_mm = None, None, None, None
        if isinstance(synthetic, tuple):
            self.syn_path, self.syn_idx = str(synthetic[0]), np.asarray(synthetic[1])
            n_syn = len(self.syn_idx)
        elif synthetic is not None:
            self.syn = synthetic
            n_syn = len(synthetic)
        else:
            n_syn = 0
        self.n_real = len(self.idx)
        self.tf = build_transform(cfg, train)
        self.labels = np.concatenate([cache.labels[self.idx], np.ones(n_syn, dtype=cache.labels.dtype)])

    def __getstate__(self):
        st = self.__dict__.copy()
        st["_syn_mm"] = None
        return st

    def _synthetic(self, k):
        if self.syn_path is None:
            return self.syn[k]
        if self._syn_mm is None:
            self._syn_mm = np.load(self.syn_path, mmap_mode="r")
        return self._syn_mm[int(self.syn_idx[k])]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        if i < self.n_real:
            j = int(self.idx[i])
            arr = np.array(self.cache.images[j])
        else:
            j = -1
            arr = np.array(self._synthetic(i - self.n_real))
        x = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
        return self.tf(x), int(self.labels[i]), j
'''

larynx_data = install_module("larynx_data", DATA_SRC)
Cache, ImageDS = larynx_data.Cache, larynx_data.ImageDS


def find_cache(cfg: dict) -> Path:
    cur = P(cfg, "cache", "current.json")
    if not cur.exists():
        raise FileNotFoundError("Cache belum dibuat (jalankan notebook 02)")
    return Path(json.loads(cur.read_text(encoding="utf-8"))["dir"])

## 3. Arsitektur GAN (sama dengan notebook 00)

In [ ]:
# ---------------------------------------------------------------- jaringan
def _resolutions(resolution: int) -> list:
    r, out = 4, []
    while r <= resolution:
        out.append(r); r *= 2
    return out


class ModConv(nn.Module):
    def __init__(self, cin, cout, w_dim, k=3, demod=True, up=False):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(cout, cin, k, k) / math.sqrt(cin * k * k))
        self.affine = nn.Linear(w_dim, cin)
        nn.init.constant_(self.affine.bias, 1.0)
        self.k, self.demod, self.up, self.cout = k, demod, up, cout

    def forward(self, x, w):
        B, C, H, W = x.shape
        s = self.affine(w)
        wt = self.weight[None] * s[:, None, :, None, None]
        if self.demod:
            wt = wt * torch.rsqrt(wt.square().sum(dim=(2, 3, 4), keepdim=True) + 1e-8)
        if self.up:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            H, W = H * 2, W * 2
        x = x.reshape(1, B * C, H, W)
        out = F.conv2d(x, wt.reshape(B * self.cout, C, self.k, self.k), padding=self.k // 2, groups=B)
        return out.reshape(B, self.cout, H, W)


class SynthLayer(nn.Module):
    def __init__(self, cin, cout, w_dim, up=False):
        super().__init__()
        self.conv = ModConv(cin, cout, w_dim, 3, True, up)
        self.noise_strength = nn.Parameter(torch.zeros(1))
        self.bias = nn.Parameter(torch.zeros(cout))

    def forward(self, x, w):
        x = self.conv(x, w)
        x = x + torch.randn(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype) * self.noise_strength
        return F.leaky_relu(x + self.bias.view(1, -1, 1, 1), 0.2)


class Generator(nn.Module):
    def __init__(self, z_dim, w_dim, n_classes, channels, resolution, map_layers):
        super().__init__()
        self.z_dim, self.n_classes, self.resolution = z_dim, n_classes, resolution
        self.embed = nn.Linear(n_classes, w_dim)
        layers, d = [], z_dim + w_dim
        for _ in range(map_layers):
            layers += [nn.Linear(d, w_dim), nn.LeakyReLU(0.2)]
            d = w_dim
        self.mapping = nn.Sequential(*layers)
        ch = {int(k): v for k, v in channels.items()}
        self.const = nn.Parameter(torch.randn(ch[4], 4, 4))
        self.res = _resolutions(resolution)
        self.conv0, self.conv1, self.torgb = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
        for r in self.res:
            if r > 4:
                self.conv0[str(r)] = SynthLayer(ch[r // 2], ch[r], w_dim, up=True)
            self.conv1[str(r)] = SynthLayer(ch[r], ch[r], w_dim)
            self.torgb[str(r)] = ModConv(ch[r], 3, w_dim, k=1, demod=False)

    def forward(self, z, y):
        z = z * torch.rsqrt(z.square().mean(dim=1, keepdim=True) + 1e-8)
        c = self.embed(F.one_hot(y, self.n_classes).to(z.dtype))
        c = c * torch.rsqrt(c.square().mean(dim=1, keepdim=True) + 1e-8)
        w = self.mapping(torch.cat([z, c], dim=1))
        x = self.const[None].expand(z.shape[0], -1, -1, -1)
        img = None
        for r in self.res:
            if r > 4:
                x = self.conv0[str(r)](x, w)
            x = self.conv1[str(r)](x, w)
            rgb = self.torgb[str(r)](x, w)
            img = rgb if img is None else F.interpolate(img, scale_factor=2, mode="bilinear", align_corners=False) + rgb
        return img


class DBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.c0 = nn.Conv2d(cin, cin, 3, padding=1)
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1, bias=False)

    def forward(self, x):
        y = F.avg_pool2d(F.leaky_relu(self.c1(F.leaky_relu(self.c0(x), 0.2)), 0.2), 2)
        return (self.skip(F.avg_pool2d(x, 2)) + y) / math.sqrt(2)


def minibatch_std(x, group=4):
    B, C, H, W = x.shape
    G = min(group, B)
    while B % G:
        G -= 1
    y = x.reshape(G, -1, C, H, W)
    y = y - y.mean(dim=0)
    y = (y.square().mean(dim=0) + 1e-8).sqrt().mean(dim=(1, 2, 3))
    return torch.cat([x, y.reshape(-1, 1, 1, 1).repeat(G, 1, H, W)], dim=1)


class Discriminator(nn.Module):
    def __init__(self, in_ch, n_classes, channels, resolution):
        super().__init__()
        ch = {int(k): v for k, v in channels.items()}
        self.res = _resolutions(resolution)[::-1]
        self.fromrgb = nn.Conv2d(in_ch, ch[self.res[0]], 1)
        self.blocks = nn.ModuleList([DBlock(ch[r], ch[r // 2]) for r in self.res[:-1]])
        c4 = ch[4]
        self.conv = nn.Conv2d(c4 + 1, c4, 3, padding=1)
        self.fc = nn.Linear(c4 * 16, c4)
        self.out = nn.Linear(c4, 1)
        self.embed = nn.Embedding(n_classes, c4)
        self.c4 = c4

    def forward(self, x, y):
        h = F.leaky_relu(self.fromrgb(x), 0.2)
        for b in self.blocks:
            h = b(h)
        h = F.leaky_relu(self.conv(minibatch_std(h)), 0.2)
        h = F.leaky_relu(self.fc(h.flatten(1)), 0.2)
        return self.out(h) + (h * self.embed(y)).sum(dim=1, keepdim=True) / math.sqrt(self.c4)


# ---------------------------------------------------------------- Frangi diferensiabel untuk D_roi
def _kernels(sigma, device, dtype):
    r = int(math.ceil(3 * sigma))
    x = torch.arange(-r, r + 1, device=device, dtype=dtype)
    g = torch.exp(-x ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return g, -x / sigma ** 2 * g, (x ** 2 / sigma ** 4 - 1 / sigma ** 2) * g, r


def _sep(img, kx, ky, r):
    x = F.pad(img, (r, r, r, r), mode="reflect")
    return F.conv2d(F.conv2d(x, kx.view(1, 1, 1, -1)), ky.view(1, 1, -1, 1))


def _blur(img, sigma):
    if sigma <= 0:
        return img
    g, _, _, r = _kernels(sigma, img.device, img.dtype)
    return _sep(img, g, g, r)


def fov_mask(rgb, thr, erode):
    with torch.no_grad():
        m = (rgb.max(dim=1, keepdim=True).values > thr).float()
        if erode > 0:
            m = -F.max_pool2d(-m, 2 * erode + 1, stride=1, padding=erode)
    return m


def frangi_soft(gray, v):
    gray = _blur(gray, v["blur"])
    best = None
    for s in v["sigmas"]:
        g, g1, g2, r = _kernels(float(s), gray.device, gray.dtype)
        hxx, hyy, hxy = _sep(gray, g2, g, r) * s ** 2, _sep(gray, g, g2, r) * s ** 2, _sep(gray, g1, g1, r) * s ** 2
        tmp = torch.sqrt(((hxx - hyy) / 2) ** 2 + hxy ** 2 + 1e-12)
        l1, l2 = (hxx + hyy) / 2 + tmp, (hxx + hyy) / 2 - tmp
        small = torch.where(l1.abs() < l2.abs(), l1, l2)
        large = torch.where(l1.abs() < l2.abs(), l2, l1)
        rb = small.abs() / (large.abs() + 1e-8)
        s2 = small ** 2 + large ** 2
        resp = torch.exp(-rb ** 2 / (2 * v["beta"] ** 2)) * (1 - torch.exp(-s2 / (2 * v["c"] ** 2)))
        resp = torch.where(large > 0, resp, torch.zeros_like(resp))
        best = resp if best is None else torch.maximum(best, resp)
    return best


def roi_soft(rgb, v):
    fov = fov_mask(rgb, v["fov_thr"], v["fov_erode"])
    green = (rgb[:, 1:2] + 1) / 2
    mean = (green * fov).sum(dim=(1, 2, 3), keepdim=True) / (fov.sum(dim=(1, 2, 3), keepdim=True) + 1e-6)
    green = torch.where(fov > 0, green, mean.expand_as(green))
    resp = frangi_soft(green, v) / v["norm"]
    return torch.sigmoid((resp - v["threshold"]) / v["temperature"]) * fov


def roi_input(rgb, roi):
    return torch.cat([(rgb + 1) * roi - 1, roi * 2 - 1], dim=1)


# ---------------------------------------------------------------- augmentasi ADA-lite
def augment(x, p, color=True):
    if p <= 0:
        return x
    B, dev = x.shape[0], x.device

    def flag(shape=(B, 1, 1, 1)):
        return (torch.rand(shape, device=dev) < p).to(x.dtype)

    x = torch.where(flag().bool(), x.transpose(2, 3), x)
    fx, fy = 1 - 2 * flag((B,)), 1 - 2 * flag((B,))
    tx = (torch.rand(B, device=dev) * 2 - 1) * 0.25 * flag((B,))
    ty = (torch.rand(B, device=dev) * 2 - 1) * 0.25 * flag((B,))
    theta = torch.zeros(B, 2, 3, device=dev, dtype=x.dtype)
    theta[:, 0, 0], theta[:, 1, 1], theta[:, 0, 2], theta[:, 1, 2] = fx, fy, tx, ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    if color:
        x = x + torch.randn(B, 1, 1, 1, device=dev, dtype=x.dtype) * 0.2 * flag()
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - mean) * torch.exp2(torch.randn(B, 1, 1, 1, device=dev, dtype=x.dtype) * 0.5 * flag()) + mean
    return x


def build_nets(cfg: dict, lam: float, device: str):
    g = cfg["gan"]
    G = Generator(g["z_dim"], g["w_dim"], 2, g["channels"], g["resolution"], g["map_layers"]).to(device)
    D = Discriminator(3, 2, g["d_channels"], g["resolution"]).to(device)
    droi_ch = g.get("droi_d_channels") or g["d_channels"]          # Versi 2: D_roi lebih ramping
    Droi = Discriminator(4, 2, droi_ch, g["resolution"]).to(device) if lam > 0 else None
    return G, D, Droi


_g = build_nets(cfg, 1.0, "cpu")
print("parameter G / D / D_roi:", [f"{sum(p.numel() for p in m.parameters()) / 1e6:.1f} jt" for m in _g])
del _g

## 4. Kode pelatihan GAN, metrik kualitas, dan pilot (dari `train_gan.py`, `gan/metrics.py`, `pilot_lambda.py`)

In [ ]:
# ---------------------------------------------------------------- pelatihan GAN (dari train_gan.py, gan/metrics.py, pilot_lambda.py Versi 1)
# Perbedaan Versi 2: gradient accumulation lewat gan.micro_batch (batch efektif tetap gan.batch_size).
from scipy import linalg
from scipy.stats import wasserstein_distance


# ------------------------------------------------------------------ util data
def to_float(u8: torch.Tensor, device: str) -> torch.Tensor:
    return u8.to(device, non_blocking=True).float() / 127.5 - 1.0


def load_u8(cache, idx: np.ndarray, res: int) -> torch.Tensor:
    # (N,3,res,res) uint8 di CPU. Diperkecil dengan area-interpolasi bila res != ukuran cache.
    out = []
    for s in range(0, len(idx), 256):
        a = torch.from_numpy(np.array(cache.images[np.sort(idx[s:s + 256])])).permute(0, 3, 1, 2)
        if a.shape[-1] != res:
            a = F.interpolate(a.float(), size=(res, res), mode="area").round().clamp(0, 255).to(torch.uint8)
        out.append(a)
    return torch.cat(out) if out else torch.zeros(0, 3, res, res, dtype=torch.uint8)


@torch.no_grad()
def sample_u8(G, n: int, label, labels_pool, bs: int, device: str, seed: int, out_size=None) -> np.ndarray:
    # Bangkitkan n citra (N,H,W,3) uint8. label=None -> label diambil dari labels_pool (distribusi empiris).
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    out = []
    for s in range(0, n, bs):
        b = min(bs, n - s)
        z = torch.randn(b, G.z_dim, generator=g).to(device)
        y = (torch.full((b,), label, dtype=torch.long) if label is not None
             else labels_pool[torch.randint(len(labels_pool), (b,), generator=g)]).to(device)
        x = G(z, y).clamp(-1, 1)
        if out_size and x.shape[-1] != out_size:
            x = F.interpolate(x, size=(out_size, out_size), mode="bilinear", align_corners=False)
        out.append(((x + 1) * 127.5).round().clamp(0, 255).to(torch.uint8).permute(0, 2, 3, 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 1, 1, 3), np.uint8)


def atomic_save_npy(path: Path, arr: np.ndarray) -> None:
    # .npy tak terkompresi supaya dapat dibuka dengan memmap oleh pekerja DataLoader (citra GAN hampir tidak terkompresi).
    buf = io.BytesIO()
    np.save(buf, arr)
    atomic_write_bytes(path, buf.getvalue())


# ------------------------------------------------------------------ metrik kualitas (FR-11)
# FID memakai Inception-v3 ImageNet torchvision (bukan TF-Inception standar): tidak sebanding langsung dengan literatur (PRD §11).
def load_inception(device: str):
    from torchvision.models import Inception_V3_Weights, inception_v3

    m = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True, transform_input=False)
    m.fc = torch.nn.Identity()
    return m.eval().to(device)


@torch.no_grad()
def inception_feats(model, imgs_u8: np.ndarray, device: str, bs: int = 64) -> np.ndarray:
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(imgs_u8), bs):
        x = torch.from_numpy(np.ascontiguousarray(imgs_u8[i:i + bs])).to(device).permute(0, 3, 1, 2).float() / 255
        x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        out.append(model((x - mean) / std).float().cpu().numpy())
    return np.concatenate(out)


def fid_from_feats(a: np.ndarray, b: np.ndarray) -> float:
    mu1, mu2 = a.mean(0), b.mean(0)
    s1, s2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    covmean = linalg.sqrtm(s1 @ s2)      # tanpa disp=False: argumen itu usang (SciPy 1.17) dan dihapus di SciPy 1.18
    if not np.isfinite(covmean).all():
        off = np.eye(s1.shape[0]) * 1e-6
        covmean = linalg.sqrtm((s1 + off) @ (s2 + off))
    covmean = covmean.real
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))


def vessel_fractions(imgs_u8: np.ndarray, q: dict) -> np.ndarray:
    # Fraksi area pembuluh per citra dengan filter SATO pada kanal hijau di dalam area pandang (L6: berbeda dari Frangi).
    from scipy import ndimage as ndi
    from skimage.filters import sato

    out = []
    for im in imgs_u8:
        g = im[..., 1].astype(np.float32) / 255.0
        fov = ndi.binary_erosion(im.max(axis=2) > 0, iterations=4)
        if fov.sum() < 100:
            out.append(0.0)
            continue
        g = np.where(fov, g, g[fov].mean())
        v = sato(g, sigmas=q["vessel_sigmas"], black_ridges=True)
        out.append(float(((v > q["vessel_threshold"]) & fov).sum() / fov.sum()))
    return np.asarray(out)


def vessel_w1(real_u8: np.ndarray, fake_u8: np.ndarray, q: dict) -> float:
    return float(wasserstein_distance(vessel_fractions(real_u8, q), vessel_fractions(fake_u8, q)))


@torch.no_grad()
def lpips_diversity(imgs_u8: np.ndarray, n_pairs: int, net: str, device: str, seed: int) -> float:
    # Rata-rata LPIPS antar pasangan acak citra sintetis sekelas (lebih tinggi = lebih beragam).
    import lpips

    if len(imgs_u8) < 2:
        return float("nan")
    fn = lpips.LPIPS(net=net, verbose=False).to(device).eval()
    rng = np.random.default_rng(seed)
    i = rng.integers(0, len(imgs_u8), n_pairs)
    j = (i + rng.integers(1, len(imgs_u8), n_pairs)) % len(imgs_u8)
    vals = []
    for s in range(0, n_pairs, 32):
        a = torch.from_numpy(imgs_u8[i[s:s + 32]]).to(device).permute(0, 3, 1, 2).float() / 127.5 - 1
        b = torch.from_numpy(imgs_u8[j[s:s + 32]]).to(device).permute(0, 3, 1, 2).float() / 127.5 - 1
        vals.append(fn(a, b).flatten().cpu().numpy())
    del fn
    return float(np.concatenate(vals).mean())


def save_grid(imgs_u8: np.ndarray, path, n_col: int = 10) -> None:
    import matplotlib.pyplot as plt

    n = len(imgs_u8)
    n_row = -(-n // n_col)
    fig, axes = plt.subplots(n_row, n_col, figsize=(1.6 * n_col, 1.6 * n_row), squeeze=False)
    for k, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if k < n:
            ax.imshow(imgs_u8[k])
    fig.tight_layout(pad=0.1)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=80)
    plt.close(fig)


# ------------------------------------------------------------------ inti pelatihan
def train_core(cfg: dict, real_u8: torch.Tensor, labels: torch.Tensor, val_u8, lam: float, total_kimg: float,
               d: Path, h: str, seed: int, device: str) -> dict:
    g = cfg["gan"]
    B = g["batch_size"]
    micro = int(g.get("micro_batch") or B)
    if B % micro:
        raise ValueError(f"gan.batch_size {B} harus kelipatan gan.micro_batch {micro}")
    n_acc = B // micro
    G, D, Droi = build_nets(cfg, lam, device)
    G_ema = copy.deepcopy(G).eval().requires_grad_(False)
    optG = torch.optim.Adam(G.parameters(), lr=g["lr_g"], betas=tuple(g["betas"]))
    optD = torch.optim.Adam(list(D.parameters()) + (list(Droi.parameters()) if Droi is not None else []),
                            lr=g["lr_d"], betas=tuple(g["betas"]))
    n_iters = int(total_kimg * 1000 / B)
    snap_every = max(1, int(g["snapshot_kimg"] * 1000 / B))
    ada, v = g["ada"], g["vesselness"]
    gam = g["r1_gamma"] / 2 * g["r1_interval"]
    sp = F.softplus
    st = {"it": 0, "p": 0.0, "history": []}
    last = d / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck.get("config_hash") == h:
            G.load_state_dict(ck["G"]); G_ema.load_state_dict(ck["G_ema"]); D.load_state_dict(ck["D"])
            if Droi is not None and ck.get("Droi"):
                Droi.load_state_dict(ck["Droi"])
            optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"])
            st = {k: ck[k] for k in ("it", "p", "history")}
            torch.set_rng_state(ck["rng_torch"]); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
            print(f"[lanjut] GAN dari iterasi {st['it']}/{n_iters}")
        else:
            print("[peringatan] last.pt GAN dari konfigurasi lain diabaikan")

    inc = load_inception(device) if val_u8 is not None else None
    val_feats = inception_feats(inc, val_u8, device) if inc is not None else None
    ada_stat = []
    N = len(real_u8)
    labels = labels.long()
    t0 = time.time()
    for it in range(st["it"], n_iters):
        p = st["p"] if ada["enabled"] else 0.0
        idx = torch.randint(N, (B,))
        real_all, y_all = to_float(real_u8[idx], device), labels[idx].to(device)
        yg_all = labels[torch.randint(N, (B,))].to(device)
        do_r1 = it % g["r1_interval"] == 0
        # ---- langkah D (gradient accumulation per potongan micro_batch)
        G.requires_grad_(False); D.requires_grad_(True)
        if Droi is not None:
            Droi.requires_grad_(True)
        optD.zero_grad(set_to_none=True)
        loss_D_val, sign_val = 0.0, 0.0
        for s in range(0, B, micro):
            xr, yy, yg = real_all[s:s + micro], y_all[s:s + micro], yg_all[s:s + micro]
            with torch.no_grad():
                fake = G(torch.randn(micro, G.z_dim, device=device), yg)
            d_real = D(augment(xr, p), yy)
            loss = sp(D(augment(fake, p), yg)).mean() + sp(-d_real).mean()
            rr = None
            if Droi is not None:
                with torch.no_grad():
                    rr, rf = roi_input(xr, roi_soft(xr, v)), roi_input(fake, roi_soft(fake, v))
                loss = loss + sp(Droi(augment(rf, p, color=False), yg)).mean() + sp(-Droi(augment(rr, p, color=False), yy)).mean()
            (loss / n_acc).backward()
            loss_D_val += float(loss) / n_acc
            sign_val += float(torch.sign(d_real).mean()) / n_acc
            if do_r1:        # R1 lazy
                ra = augment(xr, p).detach().requires_grad_(True)
                gr, = torch.autograd.grad(D(ra, yy).sum(), ra, create_graph=True)
                r1 = gr.square().sum(dim=(1, 2, 3)).mean() * gam
                if Droi is not None:
                    xa = augment(rr, p, color=False).detach().requires_grad_(True)
                    gr2, = torch.autograd.grad(Droi(xa, yy).sum(), xa, create_graph=True)
                    r1 = r1 + gr2.square().sum(dim=(1, 2, 3)).mean() * gam
                (r1 / n_acc).backward()
        optD.step()
        # ---- langkah G
        G.requires_grad_(True); D.requires_grad_(False)
        if Droi is not None:
            Droi.requires_grad_(False)
        optG.zero_grad(set_to_none=True)
        loss_G_val = 0.0
        for s in range(0, B, micro):
            yg = yg_all[s:s + micro]
            fake = G(torch.randn(micro, G.z_dim, device=device), yg)
            lg = sp(-D(augment(fake, p), yg)).mean()
            if Droi is not None:
                lg = lg + lam * sp(-Droi(augment(roi_input(fake, roi_soft(fake, v)), p, color=False), yg)).mean()
            (lg / n_acc).backward()
            loss_G_val += float(lg) / n_acc
        optG.step()
        # ---- EMA dan ADA
        with torch.no_grad():
            beta = 0.5 ** (B / (g["ema_kimg"] * 1000))
            for pe, pg in zip(G_ema.parameters(), G.parameters()):
                pe.copy_(pg.lerp(pe, beta))
            ada_stat.append(sign_val)
            if ada["enabled"] and (it + 1) % ada["interval"] == 0:
                rt = float(np.mean(ada_stat))
                st["p"] = float(min(1.0, max(0.0, st["p"] + np.sign(rt - ada["target"]) * B * ada["interval"] / (ada["kimg"] * 1000))))
                ada_stat = []
        st["it"] = it + 1
        if st["it"] % snap_every == 0 or st["it"] == n_iters:
            rec = {"kimg": st["it"] * B / 1000, "loss_D": loss_D_val, "loss_G": loss_G_val, "ada_p": st["p"], "seconds": time.time() - t0}
            if inc is not None:
                n = min(g["fid_samples"], len(val_u8))
                fk = sample_u8(G_ema, n, None, labels, 32, device, seed + st["it"])
                rec["fid_val"] = fid_from_feats(val_feats[:n] if n < len(val_feats) else val_feats, inception_feats(inc, fk, device))
            st["history"].append(rec)
            print(rec, flush=True)
            atomic_torch_save({"G": G.state_dict(), "G_ema": G_ema.state_dict(), "D": D.state_dict(),
                               "Droi": Droi.state_dict() if Droi is not None else None, "optG": optG.state_dict(),
                               "optD": optD.state_dict(), "config_hash": h, "rng_torch": torch.get_rng_state(),
                               "rng_np": np.random.get_state(), "rng_py": random.getstate(), **st}, last)
            if device == "cuda":
                torch.cuda.empty_cache()
    atomic_torch_save(G_ema.state_dict(), d / "G_ema.pt")
    return {"G_ema": G_ema, "history": st["history"], "labels": labels, "inception": inc}


def quality_report(cfg: dict, G_ema, real_u8_np: np.ndarray, real_labels: np.ndarray, device: str, seed: int, out_dir, inc=None) -> dict:
    # FID, W1 pembuluh (Sato), LPIPS terhadap citra asli real_u8_np (validasi pada pilot, uji pada pelaporan).
    check_l6(cfg)
    q, size = cfg["gan"]["quality"], real_u8_np.shape[1]
    inc = inc or load_inception(device)
    fr_all = inception_feats(inc, real_u8_np, device)
    pool = torch.as_tensor(real_labels)
    # Kovarians Inception 2048-dim dari <2048 citra nyata singular -> FID bias dan berderau. Citra palsu diperbanyak
    # (gan.fid_fake_samples) dan FID dirata-ratakan atas 3 seed pembangkitan; simpangan baku dilaporkan (turunan derau pembangkitan
    # saja, BUKAN derau pelatihan). Untuk pembandingan antar lambda, himpunan nyata yang dipakai sama.
    n_fake = max(len(real_u8_np), int(cfg["gan"].get("fid_fake_samples", 5000)))
    fids = [fid_from_feats(fr_all, inception_feats(inc, sample_u8(G_ema, n_fake, None, pool, 32, device, seed + 1000 * s, size), device))
            for s in range(3)]
    rep = {"fid_all": float(np.mean(fids)), "fid_all_std": float(np.std(fids)), "n_real": int(len(real_u8_np)), "n_fake": int(n_fake),
           "fid_catatan": "kovarians singular bila n_real < 2048; nilai tidak sebanding dengan literatur"}
    for c, name in ((0, "benign"), (1, "malignant")):
        real_c = real_u8_np[real_labels == c]
        fake_c = sample_u8(G_ema, max(q["n_grid"], min(len(real_c), 500)), c, None, 32, device, seed + 10 + c, size)
        if len(real_c) > 1:
            rep[f"fid_{name}"] = fid_from_feats(inception_feats(inc, real_c, device), inception_feats(inc, fake_c, device))
            rep[f"vessel_w1_{name}"] = vessel_w1(real_c[:300], fake_c[:300], q)
        rep[f"lpips_diversity_{name}"] = lpips_diversity(fake_c, q["lpips_pairs"], q["lpips_net"], device, seed)
        if out_dir is not None:
            save_grid(fake_c[:q["n_grid"]], Path(out_dir) / f"samples_{name}.png")
    return rep


def resolve_lambda(cfg: dict) -> float:
    lam = cfg["gan"]["lambda_roi"]
    if lam is None:
        sp_ = P(cfg, "cp4", "lambda_selection.json")
        if not sp_.exists():
            raise FileNotFoundError(f"lambda_roi belum ditetapkan: jalankan pilot (D2) lebih dulu -> {sp_}")
        lam = json.loads(sp_.read_text(encoding="utf-8"))["chosen"]
    return float(lam)


def run_gan_job(cfg: dict, rep: int, fold: int, lam: float, d: Path, mode: str = "full", cache_dir=None, total_kimg=None) -> dict:
    # mode='full' (S5/S6): latih, bangkitkan sintetis, laporan terhadap UJI. mode='pilot': latih, laporan terhadap VALIDASI.
    splits = load_splits(cfg)
    check_splits(splits, resolution_map(cfg) if splits.get("kind") == "s7" else None)
    check_l6(cfg)
    cache = Cache(cache_dir or find_cache(cfg))
    sf = get_fold(splits, rep, fold)
    tk = total_kimg or cfg["gan"]["total_kimg"]
    h = config_hash({"cfg": cfg, "rep": rep, "fold": fold, "lam": lam, "mode": mode, "tk": tk, "cache": cache.hash,
                     "splits": splits["meta"]["hash"]})
    if is_done(d, h):
        print(f"[dilewati] GAN {d} sudah DONE")
        return json.loads((d / "gan_metrics.json").read_text(encoding="utf-8"))
    d.mkdir(parents=True, exist_ok=True)
    atomic_write_json(d / "patients.json", {"train": sf["train"], "val": sf["val"], "test": sf["test"]})
    check_l3_job(json.loads((d / "patients.json").read_text(encoding="utf-8")), sf)

    seed = job_seed(cfg["seed"], cfg["scenario"], "gan", rep, fold, lam)
    set_seed(seed)
    device = DEV
    torch.backends.cudnn.benchmark = True
    res = cfg["gan"]["resolution"]
    idx_tr, idx_va = np.sort(cache.indices_of(sf["train"])), np.sort(cache.indices_of(sf["val"]))
    real_u8 = load_u8(cache, idx_tr, res)
    labels = torch.from_numpy(cache.labels[idx_tr])
    val_np = np.array(cache.images[idx_va])
    if val_np.shape[1] != res:
        val_np = F.interpolate(torch.from_numpy(val_np).permute(0, 3, 1, 2).float(), size=(res, res), mode="area").round().to(torch.uint8).permute(0, 2, 3, 1).numpy()
    write_run_info(d, cfg, h, seed, {"rep": rep, "fold": fold, "lambda_roi": lam, "mode": mode, "total_kimg": tk, "device": device,
                                     "micro_batch": cfg["gan"].get("micro_batch"), "cache_hash": cache.hash,
                                     "splits_hash": splits["meta"]["hash"], "n_train_images": int(len(idx_tr)), "n_val_images": int(len(idx_va)),
                                     "catatan": "GAN ringkas bergaya StyleGAN2, ADA-lite (decisions.yaml D8), gradient accumulation Versi 2"})
    out = train_core(cfg, real_u8, labels, val_np, lam, tk, d, h, seed, device)
    G_ema, inc = out["G_ema"], out["inception"]
    atomic_write_bytes(d / "gan_history.json", json.dumps(out["history"], indent=2).encode("utf-8"))

    if mode == "pilot":
        metrics = quality_report(cfg, G_ema, val_np, cache.labels[idx_va], device, seed, d / "grid_val", inc)
        metrics["evaluated_on"] = "pasien validasi dalam lipatan latih (BUKAN uji)"
    else:
        n_syn = int(round(cfg["gan"]["generate_fraction"] * int((cache.labels[idx_tr] == 1).sum())))
        syn = sample_u8(G_ema, n_syn, 1, None, 32, device, seed + 99, 256)
        atomic_save_npy(d / "synth.npy", syn)
        atomic_write_json(d / "synth_meta.json", {"n": int(n_syn), "class": 1, "source_patients": sf["train"],
                                                  "lambda_roi": lam, "seed": seed + 99, "never_in_val_or_test": True})
        idx_te = np.sort(cache.indices_of(sf["test"]))
        test_np = np.array(cache.images[idx_te])
        metrics = quality_report(cfg, G_ema, test_np, cache.labels[idx_te], device, seed, d / "grid_test", inc)
        metrics["evaluated_on"] = "citra uji lipatan ini, UNTUK PELAPORAN (FR-11), tidak dipakai memilih apa pun"
        metrics["n_synthetic_generated"] = int(n_syn)
    metrics["lambda_roi"] = lam
    metrics["final_fid_val_monitor"] = out["history"][-1].get("fid_val") if out["history"] else None
    atomic_write_json(d / "gan_metrics.json", metrics)
    mark_done(d, h)
    del out, G_ema, inc, real_u8
    return metrics


def gan_job_cli(cfg: dict, rep: int, fold: int) -> dict:
    # S5/S6/S7: lambda_roi dari konfigurasi atau hasil pilot (CP4).
    lam = resolve_lambda(cfg)
    cfg["gan"]["lambda_roi"] = lam
    return run_gan_job(cfg, rep, fold, lam, job_dir(cfg, f"{cfg['scenario']}_gan", rep, fold))


def run_pilot(cfg: dict) -> dict:
    # D2 / CP4: pilot lambda_roi pada satu lipatan, DIPILIH dari pasien VALIDASI di dalam lipatan latih (FID validasi terendah).
    pl = cfg["pilot"]
    if pl["criterion"] != "val_fid":
        raise ValueError("kriteria pilot yang didukung: val_fid")
    rows = []
    for lam in pl["lambdas"]:
        c = copy.deepcopy(cfg)
        c["gan"]["lambda_roi"] = lam
        d = job_dir(c, "pilot", pl["rep"], pl["fold"]) / f"lam{lam}"
        m = run_job_safe(f"pilot/lam{lam}", lambda c=c, d=d, lam=lam: run_gan_job(c, pl["rep"], pl["fold"], float(lam), d, mode="pilot", total_kimg=pl["total_kimg"]), cfg)
        rows.append({"lambda_roi": lam, "fid_val": m["fid_all"], "fid_val_std": m.get("fid_all_std"), "fid_val_malignant": m.get("fid_malignant"),
                     "vessel_w1_malignant": m.get("vessel_w1_malignant"),
                     "lpips_diversity_malignant": m.get("lpips_diversity_malignant"), "dir": str(d)})
    best = min(rows, key=lambda r: (r["fid_val"], r["lambda_roi"]))
    others = sorted(r["fid_val"] for r in rows if r is not best)
    noise = max(float(r.get("fid_val_std") or 0.0) for r in rows)
    gap = (others[0] - best["fid_val"]) if others else float("inf")
    sel = {"chosen": best["lambda_roi"], "criterion": pl["criterion"], "table": rows,
           "source": f"pasien validasi dalam lipatan latih (rep{pl['rep']}, lipatan {pl['fold']}); uji tidak dipakai",
           "selisih_terbaik_vs_kedua": gap, "derau_pembangkitan_maks": noise,
           "pilihan_dalam_derau": bool(gap <= 2 * noise),
           "peringatan": "FID Inception-ImageNet kurang andal pada CE-NBI (PRD §11), satu lipatan dan satu seed pelatihan; "
                         "bila pilihan_dalam_derau=true, selisih antar lambda tidak dapat dibedakan dari derau: putuskan dengan grid sampel, "
                         "LPIPS, dan metrik pembuluh, lalu catat alasannya di LOG.md"}
    atomic_write_json(P(cfg, "cp4", "lambda_selection.json"), sel)
    return sel

In [ ]:
# ---------------------------------------------------------------- pemeriksaan checkpoint (dari checkpoint.py Versi 1)
# Prinsip: item yang tidak bisa dijalankan dilaporkan apa adanya (GAGAL / TIDAK_TERSEDIA / MENUNGGU), tidak pernah diubah menjadi LULUS.
# `lulus` benar hanya bila tidak ada GAGAL atau MENUNGGU. manifest.json ditulis hanya bila lulus. Tag git TIDAK dibuat di sini.
def _scenario_items(cfg: dict, sc: str, label=None):
    # Item standar satu skenario: semua job DONE, L3 lolos, tabel hasil lengkap dengan interval.
    label = label or sc
    items, files = [], []
    st = job_status(cfg, sc)
    n_done = sum(r["DONE"] for r in st)
    items.append(_item(f"{label}: seluruh lipatan berstatus DONE", LULUS if n_done == len(st) else GAGAL, f"{n_done}/{len(st)}"))
    try:
        n = check_l3_dir(P(cfg, "jobs"), sc, load_splits(cfg))
        items.append(_item(f"{label}: L3 daftar pasien tiap job", LULUS, f"{n} job"))
    except LeakageError as e:
        items.append(_item(f"{label}: L3", GAGAL, str(e)))
    rp = P(cfg, "results", f"results_{sc}.csv")
    if rp.exists():
        r = pd.read_csv(rp)
        full = r.job_selesai.str.split("/").map(lambda a: a[0] == a[1]).all()
        items.append(_item(f"{label}: tabel hasil dengan interval kepercayaan", LULUS if full else GAGAL,
                           "; ".join(f"{x.metric}={x.estimate:.3f} [{x.ci_low:.3f}, {x.ci_high:.3f}]" for x in r.itertuples())
                           + f" | job {r.job_selesai.iloc[0]}"))
        files += [rp, P(cfg, "results", f"fold_metrics_{sc}.csv"), P(cfg, "results", f"patient_probs_{sc}.csv")]
    else:
        items.append(_item(f"{label}: tabel hasil", GAGAL, f"{rp.name} tidak ada"))
    return items, files


def finalize_cp4(cfg: dict) -> dict:
    out = P(cfg, "cp4")
    items, files = [], []
    sp_ = out / "lambda_selection.json"
    if sp_.exists():
        sel = json.loads(sp_.read_text(encoding="utf-8"))
        items.append(_item("lambda_roi dipilih dari validasi dalam lipatan latih (D2)", LULUS if "validasi" in sel["source"] else GAGAL,
                           f"terpilih={sel['chosen']}; " + "; ".join(f"lam={r['lambda_roi']}: FIDval={r['fid_val']:.1f}" for r in sel["table"])))
        files.append(sp_)
        done = all((Path(r["dir"]) / "DONE").exists() for r in sel["table"])
        items.append(_item("Semua pilot GAN berstatus DONE", LULUS if done else GAGAL, f"{len(sel['table'])} nilai lambda"))
        ok = all(r.get("lpips_diversity_malignant") is not None and np.isfinite(r["lpips_diversity_malignant"]) for r in sel["table"])
        items.append(_item("FID dan LPIPS tercatat", LULUS if ok else GAGAL, "FID validasi dan LPIPS ganas per lambda"))
        grids = [Path(r["dir"]) / "grid_val" / "samples_malignant.png" for r in sel["table"]]
        items.append(_item("100 citra sintetis per kelas dibuat untuk pemeriksaan visual", LULUS if all(g.exists() for g in grids) else GAGAL,
                           f"{sum(g.exists() for g in grids)}/{len(grids)} grid ganas"))
    else:
        items.append(_item("lambda_selection.json", GAGAL, "jalankan run_pilot(cfg)"))
    try:
        check_l6(cfg)
        items.append(_item("L6: filter metrik pembuluh (Sato) berbeda dari pembuat ROI (Frangi)", LULUS,
                           f"{cfg['gan']['quality']['vessel_filter']} vs {cfg['preprocess']['roi']['filter']}"))
    except LeakageError as e:
        items.append(_item("L6", GAGAL, str(e)))
    items.append(_flag_item(out, "GAN_VISUAL_APPROVED", "Pemeriksaan visual sampel sintetis (tanpa mode collapse jelas)",
                            "periksa grid_val/*.png, lalu: approve_flag('CP4', 'GAN_VISUAL_APPROVED', 'NAMA', 'Pemeriksaan visual sampel GAN')"))
    for d in ("D2", "D8"):
        items.append(_decision_item(d))
    items.append(_review_item(out))
    return _finish("CP4", out, items, files)

## 5. Konfigurasi GAN yang berlaku dan estimasi dari benchmark CP0

In [ ]:
bg = json.loads(P(cfg, 'cp0', 'benchmark_gan.json').read_text(encoding='utf-8'))
print('total_kimg :', cfg['gan']['total_kimg'], '| batch efektif:', cfg['gan']['batch_size'], '| micro_batch:', cfg['gan'].get('micro_batch'))
print('estimasi benchmark:', json.dumps(bg.get('estimasi'), indent=2))
print('Bila terlalu lama/pendek atau micro_batch berbeda dari rekomendasi, ubah configs/base.yaml dan catat di LOG.md SEBELUM menjalankan sel berikut.')

## 6. Pilot λ_roi (dapat dilanjutkan; job DONE dilewati)

In [ ]:
cfg6 = load_merged('configs/s6.yaml')
sel = run_pilot(cfg6)
print('lambda_roi terpilih:', sel['chosen'])

## 7. Hasil pilot dan pemeriksaan visual

In [ ]:
sel = json.loads(P(cfg, 'cp4', 'lambda_selection.json').read_text(encoding='utf-8'))
print('lambda_roi terpilih:', sel['chosen'], '|', sel['source'])
display(pd.DataFrame(sel['table']))
for r in sel['table']:
    g = Path(r['dir']) / 'grid_val' / 'samples_malignant.png'
    print('lambda =', r['lambda_roi'])
    display(Image(filename=str(g)))

In [ ]:
# Setelah memeriksa grid_val/*.png: tanpa mode collapse yang jelas -> isi nama lalu jalankan.
VISUAL_APPROVER = None      # contoh: 'Wahyu'
if VISUAL_APPROVER:
    approve_flag('CP4', 'GAN_VISUAL_APPROVED', VISUAL_APPROVER, 'Pemeriksaan visual sampel GAN')

## 8. Pemeriksaan checkpoint CP4 dan `manifest.json` (sel terakhir)

In [ ]:
# Isi nama peninjau KEDUA setelah benar-benar meninjau hasil di atas (RULES §3), lalu jalankan sel ini.
REVIEWER = None      # contoh: 'Ahmad'
if REVIEWER:
    approve_review('CP4', REVIEWER)

In [ ]:
report = finalize_cp4(cfg)
assert report['checkpoint'] == 'CP4'